# Moments in Time Class Browser

Set the parameters below, then run the notebook to list dataset classes and view example videos from selected classes.

In [ ]:
from pathlib import Path
import html
import os
import random

from IPython.display import HTML, display

# Adjustable parameters
DATASET_ROOT = os.getenv("MOMENTS_ROOT", "")  # e.g. "/path/to/moments_in_time"
SPLIT = "training"  # use "." if class folders are directly under DATASET_ROOT
SELECTED_CLASSES = []  # e.g. ["running", "jumping", "eating"]
SAMPLE_CLASSES = 0  # randomly choose this many classes when SELECTED_CLASSES is empty
N_EXAMPLES = 6
MAX_CLASSES_IN_INDEX = 400
OUTPUT_HTML = "moments_in_time_gallery.html"
RANDOM_SEED = 7

VIDEO_EXTENSIONS = {".avi", ".m4v", ".mkv", ".mov", ".mp4", ".webm"}

In [ ]:
def split_root(dataset_root, split):
    if not dataset_root:
        raise ValueError("Set DATASET_ROOT or the MOMENTS_ROOT environment variable.")
    root = Path(dataset_root).expanduser().resolve()
    if split in {"", "."}:
        return root
    return root / split


def is_video(path):
    return path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS


def discover_classes(root):
    if not root.exists():
        raise FileNotFoundError(f"Dataset split directory does not exist: {root}")
    classes = {}
    for class_dir in sorted((path for path in root.iterdir() if path.is_dir()), key=lambda path: path.name.lower()):
        videos = sorted(path for path in class_dir.rglob("*") if is_video(path))
        if videos:
            classes[class_dir.name] = videos
    if not classes:
        raise RuntimeError(
            f"No class folders containing videos were found under {root}. "
            "Expected layout like <dataset-root>/<split>/<class-name>/*.mp4."
        )
    return classes


def resolve_selected_classes(classes, requested_classes):
    if not requested_classes:
        return []
    normalized = {name.lower(): name for name in classes}
    selected = []
    missing = []
    for requested in requested_classes:
        if requested in classes:
            selected.append(requested)
        elif requested.lower() in normalized:
            selected.append(normalized[requested.lower()])
        else:
            missing.append(requested)
    if missing:
        suggestions = []
        class_names = list(classes)
        for requested in missing:
            requested_lower = requested.lower()
            matches = [
                name for name in class_names
                if requested_lower in name.lower() or name.lower() in requested_lower
            ][:5]
            suggestions.append(f"{requested}: {', '.join(matches) if matches else 'no close substring matches'}")
        raise ValueError("Unknown class name(s). Suggestions: " + " | ".join(suggestions))
    return selected


def choose_classes(classes, requested_classes, sample_classes, rng):
    selected = resolve_selected_classes(classes, requested_classes)
    if selected:
        return selected
    if sample_classes <= 0:
        return []
    class_names = list(classes)
    return rng.sample(class_names, k=min(sample_classes, len(class_names)))


def video_tag(path):
    return f"""
        <figure class=\"video-card\">
            <video controls preload=\"metadata\" src=\"{path.as_uri()}\"></video>
            <figcaption title=\"{html.escape(str(path))}\">{html.escape(path.name)}</figcaption>
        </figure>
    """


def build_html(root, classes, selected_classes, n_examples, max_classes_in_index, rng):
    total_videos = sum(len(videos) for videos in classes.values())
    index_rows = [
        f"<tr><td>{html.escape(class_name)}</td><td>{len(videos)}</td></tr>"
        for class_name, videos in list(classes.items())[:max_classes_in_index]
    ]

    selected_sections = []
    for class_name in selected_classes:
        videos = classes[class_name]
        examples = rng.sample(videos, k=min(n_examples, len(videos)))
        selected_sections.append(
            f"""
            <section>
                <h2>{html.escape(class_name)} <span>{len(videos)} videos</span></h2>
                <div class=\"video-grid\">{''.join(video_tag(path) for path in examples)}</div>
            </section>
            """
        )

    if not selected_sections:
        selected_sections.append(
            """
            <section>
                <h2>No Example Classes Selected</h2>
                <p>Set SELECTED_CLASSES or SAMPLE_CLASSES in the config cell.</p>
            </section>
            """
        )

    hidden_classes = max(0, len(classes) - max_classes_in_index)
    hidden_note = (
        f"<p>{hidden_classes} additional classes are hidden by MAX_CLASSES_IN_INDEX.</p>"
        if hidden_classes else ""
    )

    return f"""<!doctype html>
<html lang=\"en\">
<head>
    <meta charset=\"utf-8\">
    <meta name=\"viewport\" content=\"width=device-width, initial-scale=1\">
    <style>
        :root {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; color: #1f2328; }}
        body {{ margin: 0; padding: 8px; background: #f7f7f5; }}
        h1 {{ margin: 0 0 8px; font-size: 28px; }}
        h2 {{ margin: 28px 0 14px; font-size: 20px; }}
        h2 span {{ color: #667085; font-size: 15px; font-weight: 500; }}
        p {{ color: #4b5563; }}
        table {{ width: 100%; border-collapse: collapse; background: white; border: 1px solid #d7dce2; }}
        th, td {{ padding: 9px 11px; border-bottom: 1px solid #e6e9ed; text-align: left; font-size: 14px; }}
        th {{ background: #eef2f6; font-weight: 650; }}
        .video-grid {{ display: grid; grid-template-columns: repeat(auto-fill, minmax(250px, 1fr)); gap: 14px; }}
        .video-card {{ margin: 0; background: white; border: 1px solid #d7dce2; border-radius: 8px; overflow: hidden; }}
        video {{ display: block; width: 100%; aspect-ratio: 16 / 9; background: #111827; }}
        figcaption {{ padding: 8px 10px; overflow: hidden; text-overflow: ellipsis; white-space: nowrap; font-size: 13px; color: #374151; }}
        code {{ background: #e8edf2; padding: 2px 5px; border-radius: 4px; }}
    </style>
</head>
<body>
    <h1>Moments in Time Gallery</h1>
    <p>{len(classes)} classes and {total_videos} videos found under <code>{html.escape(str(root))}</code>.</p>
    {''.join(selected_sections)}
    <section>
        <h2>Class Index</h2>
        {hidden_note}
        <table><thead><tr><th>Class</th><th>Videos</th></tr></thead><tbody>{''.join(index_rows)}</tbody></table>
    </section>
</body>
</html>"""

In [ ]:
rng = random.Random(RANDOM_SEED)
root = split_root(DATASET_ROOT, SPLIT)
classes = discover_classes(root)
selected_classes = choose_classes(classes, SELECTED_CLASSES, SAMPLE_CLASSES, rng)

print(f"Found {len(classes)} classes under {root}")
print(f"Total videos: {sum(len(videos) for videos in classes.values())}")
print("Selected classes:", selected_classes if selected_classes else "none")

In [ ]:
gallery_html = build_html(
    root,
    classes,
    selected_classes,
    N_EXAMPLES,
    MAX_CLASSES_IN_INDEX,
    rng,
)

output_path = Path(OUTPUT_HTML).expanduser().resolve()
output_path.write_text(gallery_html, encoding="utf-8")
print(f"Saved gallery to {output_path}")
display(HTML(gallery_html))